# DataPilot AI — Preprocessing, chunking, FAISS (Colab-ready)

**What:** Turn collected HTML (Dataset A) into cleaned documents, overlapping chunks, embeddings, and a FAISS index.

**Why:** Raw HTML includes navigation and boilerplate. RAG quality depends on clean text, stable chunk sizes, and provenance on every chunk. Starting chunk settings (~650 tokens, ~75 overlap in `config/rag.yaml`) are a **starting point**, not a claimed optimum.

**What the executed numbers mean (already on disk):**

| Stage | Result |
|-------|--------|
| Input HTML docs | 42 |
| Accepted after cleaning | **37** |
| Rejected (exact duplicates) | **5** |
| Empty / malformed | 0 |
| Chunks | **281** |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` (384-d) |

The 5 rejections are intentional: several curated Superset/Airflow topics shared the same official page. GPU is **not** required to **inspect** artefacts. Rebuilding FAISS downloads the embedding model and takes several minutes.

Charts/EDA: `notebooks/03_eda.ipynb`.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/Masters_Project"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(PROJECT_DIR)
except ImportError:
    pass

ROOT = Path.cwd()
if not (ROOT / "config" / "preprocessing.yaml").exists() and (ROOT.parent / "config" / "preprocessing.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT.resolve())

## 2. Preprocessing summary

**What:** Load `data/processed/stats/preprocessing_stats.json` produced by `scripts/preprocess_documents.py`.

**Why:** Shows data quality for assessment: duplicates removed, no empty/malformed docs, all six sources still represented.

**Meaning:** 37 accepted documents (~598k characters) is the RAG corpus. Do not treat 42 raw files as 42 unique pages after this step.

In [ ]:
import pandas as pd

pre = json.loads((ROOT / "data" / "processed" / "stats" / "preprocessing_stats.json").read_text(encoding="utf-8"))
print("input:", pre.get("total_input_documents"))
print("accepted:", pre.get("accepted_documents"))
print("rejected:", pre.get("rejected_documents"))
print("duplicates:", pre.get("duplicate_count"))
print("empty:", pre.get("empty_document_count"))
print("malformed:", pre.get("malformed_document_count"))
print("characters:", pre.get("total_characters"))
print("approx_tokens:", pre.get("total_approx_tokens"))
display(pd.Series(pre.get("documents_per_source") or {}).rename("n").to_frame())
display(pd.Series(pre.get("documents_per_category") or {}).rename("n").to_frame())

## 3. Chunking summary

**What:** Load chunk statistics from `scripts/build_chunks.py`.

**Why:** Retrieval uses chunks, not whole manuals. Overlap reduces sentences being split away from their context.

**Meaning:** **281** chunks; PostgreSQL contributes the most (108). Analytics questions in Dataset C later showed weaker retrieval — that is a coverage finding, not a reason to invent extra documents here.

In [ ]:
ch = json.loads((ROOT / "data" / "processed" / "stats" / "chunking_stats.json").read_text(encoding="utf-8"))
print("documents_chunked:", ch.get("documents_chunked"))
print("final_chunk_count:", ch.get("final_chunk_count"))
print("chunk_size_tokens:", ch.get("chunk_size_tokens"), "overlap:", ch.get("chunk_overlap_tokens"))
print("approx token mean:", (ch.get("approx_token_summary") or {}).get("mean"))
display(pd.Series(ch.get("chunks_per_source") or {}).rename("n_chunks").to_frame())

## 4. Provenance on a chunk

Every chunk keeps `source`, `url`, `title`, `document_id`, `chunk_id` so the chatbot can cite pages instead of inventing links.

In [ ]:
chunks_path = ROOT / "knowledge_base" / "chunks" / "chunks.jsonl"
with chunks_path.open(encoding="utf-8") as fh:
    first = json.loads(fh.readline())
keep = ["chunk_id", "document_id", "source", "title", "url", "category", "topic"]
print(json.dumps({k: first.get(k) for k in keep}, indent=2))
print("content_preview:", (first.get("content") or "")[:400])

## 5. Vector store

**What:** FAISS index built by `scripts/build_vector_store.py` from those chunks.

**Why:** Semantic search over MiniLM embeddings is the retriever for Systems B and C.

**Meaning:** If `faiss.index` and `chunks_metadata.jsonl` exist, RAG can run without rebuilding. Rebuild only after changing chunking or the embedding model.

In [ ]:
vs = ROOT / "knowledge_base" / "vector_store"
print("index:", (vs / "faiss.index").exists(), (vs / "faiss.index").stat().st_size if (vs / "faiss.index").exists() else None)
print("metadata:", (vs / "chunks_metadata.jsonl").exists())
cfg_path = vs / "vector_store_config.json"
if cfg_path.exists():
    print(json.dumps(json.loads(cfg_path.read_text(encoding="utf-8")), indent=2)[:1200])

## 6. Optional rebuild (slow; skip if artefacts exist)

Uncomment in order if you must regenerate. Do not invent new HTML. Collection is notebook 01.

In [ ]:
# !python scripts/preprocess_documents.py
# !python scripts/build_chunks.py
# !python scripts/build_vector_store.py

## 7. Takeaways

- Cleaning + exact-duplicate removal produced a **37-document** corpus from 42 downloads.
- **281** overlapping chunks carry URLs used later as RAG citations.
- Chunk size/overlap are configurable; optional experiments exp_04 / exp_03 were **not** required for V1.

**Next:** `notebooks/03_eda.ipynb` (plots), then `04_qlora_finetune.ipynb` (needs T4) and `05_experiments.ipynb` (executed A/B/C tables).